# PRE 18 — Preparar datos para quien los va a consumir

## Objetivo
Exponer representaciones estables para consumidores con necesidades diferentes.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "submission").is_dir())
curated = pd.DataFrame([["T1","2026-01-01","C1","Cafe","Alimentos",20],["T2","2026-01-02","C2","Pan","Alimentos",10],["T3","2026-01-03","C1","Cuaderno","Oficina",15]], columns=["transaction_id","transaction_date","customer_id","product_name","product_category","sales_amount"])
detail = curated.copy()
mart = curated.groupby("product_category", as_index=False).sales_amount.sum().sort_values("product_category")
detail.to_parquet(ROOT / "submission/sales_detail.parquet", index=False)
import sqlite3
with sqlite3.connect(ROOT / "submission/sales_serving.db") as db:
    mart.to_sql("category_sales", db, index=False, if_exists="replace")
pd.DataFrame([["sales_detail","one row per transaction","Parquet","analyst"],["category_sales","one row per category","SQLite","dashboard"]], columns=["dataset","grain","interface","consumer"]).to_csv(ROOT / "submission/serving_manifest.csv", index=False)


## Verificación
El detalle conserva el grano transaccional; el mart conserva el grano de categoría.